# vLLM inference

vllm serve Qwen/Qwen3-1.7B --port 8000 --max-model-len 8192

vllm serve Qwen/Qwen3-1.7B --port 8000 --max-model-len 8192 --quantization bitsandbytes

In [2]:
from openai import OpenAI

# those settings use vLLM server
client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

chat_response = client.chat.completions.create(
    model="",  # use the default server model
    messages=[
        {"role": "developer", "content": "You are a helpful assistant."},
        {"role": "user", "content": "How important is LLMOps on scale 0-10? "},
    ],
    max_completion_tokens=1000,
    # turn off thinking for Qwen with /no_think
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
content = chat_response.choices[0].message.content.strip()
print("Response:\n", content)


Response:
 The importance of **LLMOps** (Large Language Model Operations) on a **scale 0-10** (where 10 is the highest, indicating the most critical and complex level of importance) depends on the specific context, industry, and goals of the organization. Here's a breakdown of how LLMOps is typically evaluated on this scale:

---

### **Scale 0-10: Importance of LLMOps**

| **Scale** | **Description** | **Implication** |
|-----------|------------------|------------------|
| **0** | LLMOps is not needed or not relevant. | The organization is not using or relying on large language models (LLMs) and does not need to manage their operations. |
| **1** | LLMOps is a minor consideration or not critical. | The organization uses LLMs but does not have a formal LLMOps process, and the impact of LLMOps is minimal. |
| **2** | LLMOps is somewhat important but not critical. | The organization uses LLMs, but the LLMOps process is not well-defined, and the impact on operations is limited. |
| **3** 

## Exercise 1 (1 point)

Compare original and quantized models:

![alt text](image.png)

![alt text](image-2.png)

### 1. Available KV cache size in GB and tokens, vLLM logs it on startup:
- Original: 2.9 GiB and 27152 tokens
- Quantized: 4.73 GiB and 44304 tokens

### 2. Inference time. Prepare 10 prompts and measure the total serving time.

In [3]:
prompts = ["How important is LLMOps on scale 0-10? ", 
           "Explain what vLLM is and what is it used for.", 
           "Explain what exactly changes when we use dynamic quantization with bitesandbytes for vLLM compared to not using it.",
           "Explain what is the KV Cache concept in vLLM.",
           "Give 10 practical examples for using vLLM.",
           "What is the Maximum concurrency for tokens per request in vLLM?",
           "Prepare a plan for a weekend break in London.",
           "Do cats have sleep apnea?",
           "Living in Poland, do we have to supplement vitamin D?",
           "What are the pros and cons of raising the number of teams to 48 in the World Cup 2026. What was the main driver for FIFA to do it?"           
           ]

In [5]:
import time

start = time.perf_counter()

for prompt in prompts:
    chat_response = client.chat.completions.create(
        model="",  # use the default server model
        messages=[
            {"role": "developer", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        max_completion_tokens=1000,
        # turn off thinking for Qwen with /no_think
        extra_body={"chat_template_kwargs": {"enable_thinking": False}}
    )
    content = chat_response.choices[0].message.content.strip()
    print("\n---------------------------Response:------------------------------------\n", content)

end = time.perf_counter()

print("\nTotal Time: ", round(end - start, 2), "seconds")


---------------------------Response:------------------------------------
 The importance of **LLMOps** (Large Language Model Operations) on a **scale 0-10** (where 10 is the highest, indicating the most critical and complex level of importance) depends on the specific context, industry, and goals of the organization. Here's a breakdown of how LLMOps is typically evaluated on this scale:

---

### **Scale 0-10: Importance of LLMOps**

| **Scale** | **Description** | **Implication** |
|-----------|------------------|------------------|
| **0** | LLMOps is not needed or not relevant. | The organization is not using or relying on large language models (LLMs) and does not need to manage their operations. |
| **1** | LLMOps is a minor consideration or not critical. | The organization uses LLMs but does not have a formal LLMOps process, and the impact of LLMOps is minimal. |
| **2** | LLMOps is somewhat important but not critical. | The organization uses LLMs, but the LLMOps process is not w

![alt text](image-3.png)

![alt text](image-4.png)

### Inference Time:
- Original: 157.14 sec
- Quantized: 78.39 sec

# Tool usage

vllm serve Qwen/Qwen3-1.7B \
  --port 8000 \
  --max-model-len 8192 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes

## Exercise 2 (1 point)

Implement read_remote_csv and read_remote_parquet tools.

In [ ]:
import datetime
import json
from typing import Callable

from openai import OpenAI

import polars as pl

def make_llm_request(prompt: str) -> str:
    client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

    messages = [
        {"role": "developer", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model="",
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        resp_message = response.choices[0].message
        messages.append(resp_message.model_dump())

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Read a CSV file from a URL and return it scontent as a text",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "URL fo the CSV file."
                        },
                    },
                    "required": ["url"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Read a Parquet file from a URL and return it scontent as a text",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "URL fo the Parquet file."
                        },
                    },
                    "required": ["url"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": read_remote_csv_tool,
        "read_remote_parquet": read_remote_parquet_tool,
    }

    return tool_definitions, tool_name_to_callable


def current_date_tool() -> str:
    return datetime.date.today().isoformat()


def weather_forecast_tool(country: str, city: str, date: str) -> str:
    if country.lower() in {"united kingdom", "uk", "england"}:
        return "Fog and rain"
    else:
        return "Sunshine"
    

def read_remote_csv_tool(url: str) -> str:
    df = pl.read_csv(url)
    return df.head(60).write_csv()

def read_remote_parquet_tool(url: str) -> str:
    df = pl.read_parquet(url, n_rows=60)
    return df.write_csv()


if __name__ == "__main__":
    prompt = " Read the CSV file: https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv. How many columns are there and what they describe?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    prompt = " Read the CSV file: https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv. What is the most common toxicity type?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    prompt = " Read the Parquet file: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet . How many columns are there and what they describe?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    prompt = " Read the Parquet file: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet . What is the most common paytment method?"
    response = make_llm_request(prompt)
    print("Response:\n", response)



Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'chatcmpl-tool-84d79f8607eca71f', 'function': {'arguments': '{"url": "https://raw.githubusercontent.com/j-adamczyk/ApisTox_dataset/refs/heads/master/outputs/dataset_final.csv"}', 'name': 'read_remote_csv'}, 'type': 'function'}], 'reasoning': None}

Generated message: {'content': 'The CSV file contains the following columns:\n\n1. **Name**: The name of the compound.\n2. **CID**: The Chemical Identifier (CID) of the compound.\n3. **CAS**: The Chemical Abstracts Service (CAS) number of the compound.\n4. **SMILES**: The SMILES notation for the compound.\n5. **source**: The source of the data (e.g., ECOTOX, PPDB, BPDB).\n6. **year**: The year the data was recorded.\n7. **toxicity_type**: The type of toxicity (e.g., Contact, Oral, etc.).\n8. **herbicide**: Indicates if the compound is an herbicide.\n9. **fungicide**: Indicates if the comp

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens. However, you requested 1000 output tokens and your prompt contains at least 7193 input tokens, for a total of at least 8193 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=7193)", 'type': 'BadRequestError', 'param': 'input_tokens', 'code': 400}}

### Comments:
Works well, however due to the context limitation blocks himself pretty quickly.

# LLM security with guardrails

## Exercise 5 (1 point)

Add guardrails to our fishing fanatic LLM:

In [48]:
from dotenv import load_dotenv
from guardrails import Guard
from guardrails.hub import DetectJailbreak, RestrictToTopic
from openai import OpenAI

def make_llm_request(prompt: str) -> str:
    input_guard = Guard().use(
        DetectJailbreak(on_fail="exception", use_local=True)
        )

    try:
        input_guard.validate(prompt)
    except Exception as e:
        return f"Sorry, I cannot help you with that, reason: {e}"
    
    
    topic_input_guard = Guard().use(
        RestrictToTopic(
            valid_topics=["fish", "fishing"],
            invalid_topics=["anything not related to fish or fishing"],
            disable_classifier=False,
            disable_llm=True,
            on_fail="exception"
        )
    )

    try:
        topic_input_guard.validate(prompt)
    except Exception as e:
        return f"Sorry, I cannot help you with that, reason: {e}"


    client = OpenAI(api_key="EMPTY", base_url="http://localhost:8000/v1")

    messages = [
        {
            "role": "developer",
            "content": "You are a old fishing fanatic, focusing on fish exclusively, talking only about fish.",
        },
        {"role": "user", "content": prompt},
]

    chat_response = client.chat.completions.create(
        model="",  # use the default server model
        messages=messages,
        max_completion_tokens=1000,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    content = chat_response.choices[0].message.content.strip()

    load_dotenv()

    
    output_guard = Guard().use(RestrictToTopic(
            valid_topics=["fish", "fishing"],
            invalid_topics=["cars", "buildings", "politics", "woodworking"],
            disable_classifier=False,
            disable_llm=True,
            on_fail="exception"
            )
        )

    try:
        output_guard.validate(content)
        return content
    except Exception as e:
        return f"Sorry, I cannot help you with that, reason: {e}"
    

if __name__ == "__main__":
    prompt = "What should I have for dinner today?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    prompt = "What fish is the king of the waters?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    prompt = "Why the pike is the king of the waters?"
    response = make_llm_request(prompt)
    print("Response:\n", response)


Device set to use cpu
Device set to use cpu


Response:
 Sorry, I cannot help you with that, reason: Validation failed for field with errors: No valid topic was found.


Device set to use cpu
Device set to use cpu


Response:
 The question "What fish is the king of the waters?" is a classic riddle with a playful answer. The answer is:

**"The king of the waters is the fish that is not a fish!"**

This is a clever play on words. The fish is the king of the waters, but it's not a fish—it's a **fish** that is **not a fish**, which is a humorous twist on the riddle itself.


Device set to use cpu
Device set to use cpu


Response:
 Sorry, I cannot help you with that, reason: Validation failed for field with errors: Invalid topics found: ['buildings', 'cars', 'woodworking', 'politics']
